# Stress Testing Trading Strategy Framework with TimeGAN

This notebook implements an end-to-end pipeline for generating synthetic, regime-conditioned financial time-series data (`Normal`, `Crisis`, and `Volatile`) using **Static-Feature-Integrated TimeGAN (SF-TimeGAN)** and stress-testing algorithmic trading strategies across real and synthetic market regimes.

# 1. Data Loading and Feature Engineering

Load historical daily OHLCV data from Yahoo Finance (`yfinance`) in ascending chronological order:

- `data_name`: `^GSPC`, `AAPL`, `GOOG`, `AMZN`, `MSFT`
- `start_date`: Start date (`YYYY-MM-DD`)
- `end_date`: End date (`YYYY-MM-DD`)

Augment the 6 base features (`Adj Close`, `Close`, `High`, `Low`, `Open`, `Volume`) with 6 technical indicators:

- `Log_Return`: Daily log returns of `Close`
- `ATR`: 14-period Average True Range
- `BBW`: 20-period Bollinger Band Width
- `MACD`: Moving Average Convergence Divergence (12-EMA minus 26-EMA)
- `MACD_Signal`: 9-period EMA of `MACD`
- `RSI`: 14-period Relative Strength Index

In [ ]:
from data_loading import real_data_loading, DEFAULT_FEATURE_NAMES

data_name = "AAPL"
seq_len = 60
start_date = "2005-01-01"
end_date = "2024-11-10"

ori_data = real_data_loading(data_name, start_date, end_date)
print(f"{data_name} dataset is ready. Number of trading days: {len(ori_data)}")

**Analysis:** The raw daily OHLCV series is augmented with causal momentum and volatility indicators (`Log_Return`, `ATR`, `BBW`, `MACD`, `MACD_Signal`, `RSI`), and initial warm-up `NaN` rows are dropped so that every remaining row has a complete 12-feature representation.

In [ ]:
print(ori_data.head())

all_names = ori_data.columns.get_level_values(0)
feature_names = all_names.tolist()
print("Feature Names:", feature_names)

**Analysis:** Inspecting the header confirms the 12 features (`Adj Close`, `Close`, `High`, `Low`, `Open`, `Volume`, `Log_Return`, `ATR`, `BBW`, `MACD`, `MACD_Signal`, `RSI`) are chronologically ordered from oldest to newest.

# 2. Data Preprocessing

Slice the chronologically ordered time-series into overlapping windows of length `seq_len = 60` while preserving forward temporal ordering inside each window.

In [ ]:
from data_loading import real_data_processing

ori_data_x = real_data_processing(ori_data, seq_len, shuffle=True, seed=42)
print(f"Number of sequences: {len(ori_data_x)}, Window shape: {ori_data_x[0].shape}")

**Analysis:** Each sample in `ori_data_x` is a `(60, 12)` forward-time trajectory representing ~3 trading months of price action and technical indicators.

# 3. Market Regime Labelling (Static Features)

Label each 60-day trajectory in `ori_data_x` into one of three mutually exclusive market regimes stored in `ori_data_s`:

- **Normal** (`[1.0, 0.0, 0.0]`): Moderate drawdown ($\text{MDD} \le 20\%$) and normal volatility.
- **Crisis** (`[0.0, 1.0, 0.0]`): Severe drawdown ($\text{MDD} > 20\%$, bear-market threshold).
- **Volatile** (`[0.0, 0.0, 1.0]`): Non-crisis drawdown ($\text{MDD} \le 20\%$) with high daily return volatility ($\sigma > 0.30 / \sqrt{252} \approx 1.89\%$, equivalent to annualized VIX $> 30$).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from data_loading import label_data

ori_data_s, metrics_df = label_data(ori_data_x, feature_names, mode="fixed")

if ori_data_s:
    print(f"\nTotal length of 'ori_data_x': {len(ori_data_x)}")
    print(f"Total length of 'ori_data_s': {len(ori_data_s)}")

    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=metrics_df,
        x="volatility",
        y="mdd",
        hue="label",
        palette={"Normal": "g", "Volatile": "b", "Crisis": "r"},
        alpha=0.7,
        s=30,
    )
    plt.title("Market Regime Clusters (Labeled)", fontsize=16)
    plt.xlabel("Volatility (Std. Dev. of Log Returns)", fontsize=12)
    plt.ylabel("Maximum Drawdown (MDD)", fontsize=12)
    plt.legend(title="Regime")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.savefig("market_regime_clusters.png", bbox_inches="tight", dpi=150)
    plt.show()

**Analysis:** The two-stage regime classifier partitions historical windows into `Normal` (low volatility, low drawdown), `Volatile` (elevated volatility without a 20% crash), and `Crisis` (drawdowns exceeding 20%).

# 4. Static-Feature-Integrated TimeGAN (SF-TimeGAN)

Train TimeGAN jointly on the static regime labels (`ori_data_s`) and temporal sequences (`ori_data_x`), or generate regime-conditioned trajectories from saved checkpoints.

In [ ]:
import warnings
import numpy as np
warnings.filterwarnings("ignore")

from timegan import timegan, generate_data_from_saved_models
from metrics.discriminative_metrics import discriminative_score_metrics
from metrics.predictive_metrics import predictive_score_metrics
from metrics.visualization_metrics import visualization

**Analysis:** Core TimeGAN training, conditional generation, and evaluation modules are imported.

In [ ]:
parameters = {
    "module": "gru",
    "hidden_dim": 32,
    "num_layer": 3,
    "iterations": 8000,
    "batch_size": 128,
    "gamma_s": 100.0,
    "gamma_v": 100.0,
    "seed": 42,
}
print("Configured TimeGAN Parameters:", parameters)

**Analysis:** Hyperparameters are configured with balanced supervised and moment loss weights (`gamma_s = 100.0`, `gamma_v = 100.0`, matching Yoon et al., 2019) so adversarial distribution matching (`G_loss_U`) is not overwhelmed.

In [ ]:
# Train TimeGAN and save models to saved_models/
generated_data_s, generated_data_x = timegan(ori_data_s, ori_data_x, parameters)

**Analysis:** The 3-phase training process (1: Autoencoder reconstruction, 2: Supervised temporal transition pre-training, 3: Joint adversarial + supervised + moment matching) saves all five networks and scalers to `saved_models/`.

In [ ]:
# Generate synthetic data from saved models with financial OHLCV consistency enforced
generated_data_s, generated_data_x = generate_data_from_saved_models(
    ori_data_s, ori_data_x, enforce_consistency=True, seed=42
)
print("Finished Synthetic Data Generation.")

**Analysis:** `generate_data_from_saved_models` produces `float32` trajectories and enforces candlestick ordering (`High >= max(Open, Close) >= min(Open, Close) >= Low`) and domain bounds.

## Separating Original and Generated Data by Market Regime

In [ ]:
generated_data_x_n, generated_data_x_c, generated_data_x_v = [], [], []
for i, s_vec in enumerate(generated_data_s):
    regime_id = int(np.argmax(s_vec))
    if regime_id == 0:
        generated_data_x_n.append(generated_data_x[i])
    elif regime_id == 1:
        generated_data_x_c.append(generated_data_x[i])
    else:
        generated_data_x_v.append(generated_data_x[i])

ori_data_x_n, ori_data_x_c, ori_data_x_v = [], [], []
for i, s_vec in enumerate(ori_data_s):
    regime_id = int(np.argmax(s_vec))
    if regime_id == 0:
        ori_data_x_n.append(ori_data_x[i])
    elif regime_id == 1:
        ori_data_x_c.append(ori_data_x[i])
    else:
        ori_data_x_v.append(ori_data_x[i])

print(f"Normal   -> Generated: {len(generated_data_x_n):4d} | Original: {len(ori_data_x_n):4d}")
print(f"Crisis   -> Generated: {len(generated_data_x_c):4d} | Original: {len(ori_data_x_c):4d}")
print(f"Volatile -> Generated: {len(generated_data_x_v):4d} | Original: {len(ori_data_x_v):4d}")

**Analysis:** Both real and synthetic datasets are partitioned by their static regime vectors (`Normal`, `Crisis`, `Volatile`) for regime-specific evaluation.

In [ ]:
import pickle

with open("generated_data_x_n.pkl", "wb") as f:
    pickle.dump(generated_data_x_n, f)
with open("generated_data_x_c.pkl", "wb") as f:
    pickle.dump(generated_data_x_c, f)
with open("generated_data_x_v.pkl", "wb") as f:
    pickle.dump(generated_data_x_v, f)
with open("ori_data_x_n.pkl", "wb") as f:
    pickle.dump(ori_data_x_n, f)
with open("ori_data_x_c.pkl", "wb") as f:
    pickle.dump(ori_data_x_c, f)
with open("ori_data_x_v.pkl", "wb") as f:
    pickle.dump(ori_data_x_v, f)

**Analysis:** The separated regime datasets are serialized to `.pkl` files for reproducible evaluation and strategy stress-testing.

# 5. Loading Saved Regime Datasets & Scalers

In [ ]:
import os
import pickle
import numpy as np
from data_loading import enforce_financial_consistency, DEFAULT_FEATURE_NAMES

feature_names = DEFAULT_FEATURE_NAMES

with open("generated_data_x_n.pkl", "rb") as f:
    generated_data_x_loaded_n = enforce_financial_consistency(pickle.load(f), feature_names)
with open("generated_data_x_c.pkl", "rb") as f:
    generated_data_x_loaded_c = enforce_financial_consistency(pickle.load(f), feature_names)
with open("generated_data_x_v.pkl", "rb") as f:
    generated_data_x_loaded_v = enforce_financial_consistency(pickle.load(f), feature_names)
with open("ori_data_x_n.pkl", "rb") as f:
    ori_data_x_loaded_n = enforce_financial_consistency(pickle.load(f), feature_names)
with open("ori_data_x_c.pkl", "rb") as f:
    ori_data_x_loaded_c = enforce_financial_consistency(pickle.load(f), feature_names)
with open("ori_data_x_v.pkl", "rb") as f:
    ori_data_x_loaded_v = enforce_financial_consistency(pickle.load(f), feature_names)

scaler_path = os.path.join("saved_models", "scalers.npz")
scalers = np.load(scaler_path)
min_val_s = scalers["min_val_s"].astype(np.float32)
max_val_s = scalers["max_val_s"].astype(np.float32)
min_val_x = scalers["min_val_x"].astype(np.float32)
max_val_x = scalers["max_val_x"].astype(np.float32)

def normalize_regime_data(data_list, min_v, max_v):
    return [((np.asarray(seq, dtype=np.float32) - min_v) / (max_v + 1e-7)).astype(np.float32) for seq in data_list]

ori_data_x_norm_n = normalize_regime_data(ori_data_x_loaded_n, min_val_x, max_val_x)
generated_data_x_norm_n = normalize_regime_data(generated_data_x_loaded_n, min_val_x, max_val_x)

ori_data_x_norm_c = normalize_regime_data(ori_data_x_loaded_c, min_val_x, max_val_x)
generated_data_x_norm_c = normalize_regime_data(generated_data_x_loaded_c, min_val_x, max_val_x)

ori_data_x_norm_v = normalize_regime_data(ori_data_x_loaded_v, min_val_x, max_val_x)
generated_data_x_norm_v = normalize_regime_data(generated_data_x_loaded_v, min_val_x, max_val_x)

print("Loaded and normalized all three regimes (Normal, Crisis, Volatile).")

**Analysis:** All three regimes (`Normal`, `Crisis`, `Volatile`) are cast to `float32`, checked for OHLCV consistency, and normalized into `[0, 1]` using distinct variables so that post-hoc RNN metrics and PCA/t-SNE operate on identically scaled features without in-place mutation.

# 6. Model-Centric Evaluation of Generated Data

We evaluate each regime using:
1. **Discriminative Score (`|accuracy - 0.5|`, lower is better, `0.0` is ideal):** Post-hoc GRU classifier trained to distinguish real vs. synthetic sequences.
2. **Predictive Score (Train-on-Synthetic, Test-on-Real MAE, lower is better):** Post-hoc GRU trained on synthetic sequences to predict next-step values and tested on real sequences.
3. **PCA & t-SNE Visualizations:** 2D projections comparing real and synthetic distributions.

## 6.1 Normal Regime

In [ ]:
metric_iteration = 5
n_eval_n = min(len(ori_data_x_norm_n), len(generated_data_x_norm_n))

discriminative_score_n = [
    discriminative_score_metrics(ori_data_x_norm_n[:n_eval_n], generated_data_x_norm_n[:n_eval_n], seed=42 + i)
    for i in range(metric_iteration)
]
predictive_score_n = [
    predictive_score_metrics(ori_data_x_norm_n[:n_eval_n], generated_data_x_norm_n[:n_eval_n], seed=42 + i)
    for i in range(metric_iteration)
]

print(f"Normal Discriminative Score: {np.mean(discriminative_score_n):.4f} +/- {np.std(discriminative_score_n):.4f}")
print(f"Normal Predictive Score:     {np.mean(predictive_score_n):.4f} +/- {np.std(predictive_score_n):.4f}")

**Analysis:** Reporting `mean +/- std` across 5 independent runs quantifies both fidelity (Discriminative Score) and temporal predictability (Predictive Score) for the `Normal` market regime.

In [ ]:
visualization(ori_data_x_norm_n[:n_eval_n], generated_data_x_norm_n[:n_eval_n], "pca")
visualization(ori_data_x_norm_n[:n_eval_n], generated_data_x_norm_n[:n_eval_n], "tsne")

**Analysis:** The PCA and t-SNE plots compare the feature-averaged temporal manifold of original vs. synthetic `Normal` regime trajectories.

## 6.2 Crisis Regime

In [ ]:
n_eval_c = min(len(ori_data_x_norm_c), len(generated_data_x_norm_c))

discriminative_score_c = [
    discriminative_score_metrics(ori_data_x_norm_c[:n_eval_c], generated_data_x_norm_c[:n_eval_c], seed=42 + i)
    for i in range(metric_iteration)
]
predictive_score_c = [
    predictive_score_metrics(ori_data_x_norm_c[:n_eval_c], generated_data_x_norm_c[:n_eval_c], seed=42 + i)
    for i in range(metric_iteration)
]

print(f"Crisis Discriminative Score: {np.mean(discriminative_score_c):.4f} +/- {np.std(discriminative_score_c):.4f}")
print(f"Crisis Predictive Score:     {np.mean(predictive_score_c):.4f} +/- {np.std(predictive_score_c):.4f}")

**Analysis:** Evaluating on properly scaled `Crisis` sequences ensures `predictive_score_metrics` compares normalized targets and predictions on `[0, 1]`.

In [ ]:
visualization(ori_data_x_norm_c[:n_eval_c], generated_data_x_norm_c[:n_eval_c], "pca")
visualization(ori_data_x_norm_c[:n_eval_c], generated_data_x_norm_c[:n_eval_c], "tsne")

**Analysis:** PCA and t-SNE projections for the `Crisis` regime on normalized features prevent `Volume` from dominating the embedding.

## 6.3 Volatile Regime

In [ ]:
n_eval_v = min(len(ori_data_x_norm_v), len(generated_data_x_norm_v))

discriminative_score_v = [
    discriminative_score_metrics(ori_data_x_norm_v[:n_eval_v], generated_data_x_norm_v[:n_eval_v], seed=42 + i)
    for i in range(metric_iteration)
]
predictive_score_v = [
    predictive_score_metrics(ori_data_x_norm_v[:n_eval_v], generated_data_x_norm_v[:n_eval_v], seed=42 + i)
    for i in range(metric_iteration)
]

print(f"Volatile Discriminative Score: {np.mean(discriminative_score_v):.4f} +/- {np.std(discriminative_score_v):.4f}")
print(f"Volatile Predictive Score:     {np.mean(predictive_score_v):.4f} +/- {np.std(predictive_score_v):.4f}")

**Analysis:** The `Volatile` regime scores evaluate how well the synthetic sequences preserve high-volatility step-to-step dynamics.

In [ ]:
visualization(ori_data_x_norm_v[:n_eval_v], generated_data_x_norm_v[:n_eval_v], "pca")
visualization(ori_data_x_norm_v[:n_eval_v], generated_data_x_norm_v[:n_eval_v], "tsne")

**Analysis:** The `Volatile` regime PCA and t-SNE plots provide a distinct visual comparison specific to the `Volatile` subset.

# 7. Trading Strategy Stress-Testing & Backtesting

We now evaluate four benchmark algorithmic trading strategies across **Real** and **Synthetic** (`Normal`, `Volatile`, `Crisis`) regimes using [`backtesting.py`](backtesting.py):
1. **Buy & Hold:** Passive benchmark fully invested across each 60-day trajectory.
2. **MACD Crossover:** Trend-following strategy (Long when lagged `MACD > MACD_Signal`, else Cash).
3. **RSI Mean-Reversion:** Enters Long when lagged `RSI < 35` (oversold) and exits when `RSI > 65` (overbought).
4. **Volatility Target:** Scales trend exposure inversely with lagged Bollinger Band Width (`BBW`).

In [ ]:
from backtesting import evaluate_strategies_across_regimes, plot_stress_test_comparison

datasets_dict = {
    ("Real", "Normal"): ori_data_x_loaded_n,
    ("Synthetic", "Normal"): generated_data_x_loaded_n,
    ("Real", "Volatile"): ori_data_x_loaded_v,
    ("Synthetic", "Volatile"): generated_data_x_loaded_v,
    ("Real", "Crisis"): ori_data_x_loaded_c,
    ("Synthetic", "Crisis"): generated_data_x_loaded_c,
}

summary_df, detailed_df = evaluate_strategies_across_regimes(
    datasets_dict, feature_names=feature_names, transaction_cost_bps=5.0
)
display(summary_df)
plot_stress_test_comparison(detailed_df, save_path="stress_test_results.png", show=True)

## Summary & Conclusions

1. **Data & Chronological Integrity:** Preserving ascending chronological order from `yfinance` ensures causal technical indicators (`Log_Return`, `ATR`, `BBW`, `MACD`, `RSI`) and forward-time Maximum Drawdown (`MDD`) accurately identify `Normal`, `Volatile`, and `Crisis` regimes.
2. **TimeGAN Architecture & Training Balance:** Conditioning `Recovery` on the static latent representation $H_S$, using `softmax` + cross-entropy on one-hot static labels, computing moment losses per timestep across the batch axis, and restoring the `100.0` weight on supervised/moment losses prevent adversarial collapse.
3. **Trading Strategy Stress-Testing:** Evaluating `Buy & Hold`, `MACD Crossover`, `RSI Mean-Reversion`, and `Volatility Target` across Real and Synthetic stress regimes (`Volatile` and `Crisis`) quantifies strategy resilience via Sharpe/Sortino ratios, 90th-percentile Maximum Drawdown (`P90_MDD_Pct`), and 95% Expected Shortfall (`Mean_CVaR95_Pct`).